## Frozen Sequence Classification

In [2]:
# import libraries
import numpy as np
import pandas as pd
from tqdm import tqdm
import random

# sklearn
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score

# transformers
import torch
import torch.nn as nn
import transformers
from transformers import AutoModel, AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
from transformers import EarlyStoppingCallback

# set up roberta
from transformers import Trainer, TrainingArguments, RobertaTokenizer, RobertaModel, RobertaConfig, RobertaForSequenceClassification
from transformers import AutoTokenizer, XLMRobertaForSequenceClassification, XLMRobertaTokenizerFast

In [ ]:
# check for GPU availability
torch.cuda.is_available()

In [4]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# set random seed
seed = torch.randint(0, 1000, (1,)).item()
seed # record for replication

## Load dataset + preprocessing

In [6]:
# import the Hugging Face datasets library (pip install datasets)
from datasets import load_dataset

# download the political tweets dataset from the Hugging Face Hub
dataset = load_dataset("Jacobvs/PoliticalTweets")

# the dataset can contain several splits; take the first one
split_name = list(dataset.keys())[0]

# convert that split into a pandas DataFrame
df_labelled = dataset[split_name].to_pandas()

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

formatted_data.parquet: reconstructing file:   0%|          |  0.00B / 26.6MB            

formatted_data.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/190491 [00:00<?, ? examples/s]

In [7]:
# keep the tweet text, the party and the label, and rename 'labels' to 'label'
df_labelled = df_labelled[['text', 'party', 'labels']]
df_labelled = df_labelled.rename(columns={'labels': 'label'})

In [8]:
# check how the label maps to the party (expected: 0 = Republican, 1 = Democrat)
pd.crosstab(df_labelled['party'], df_labelled['label'])

label,0,1
party,,
Democrat,0,97901
Republican,92590,0


In [9]:
# remove possible duplicate tweets
df_labelled = df_labelled.drop_duplicates(subset=['text'], keep='first')

In [10]:
# set label as int and text as string
df_labelled['label'] = df_labelled['label'].astype('int')
df_labelled['text'] = df_labelled['text'].astype(str)

In [11]:
# optional: the dataset is large, uncomment to train on a random subset for a faster run
# df_labelled = df_labelled.sample(n=5000, random_state=seed)

# Subset (copy avoids the SettingWithCopyWarning)
df_raw = df_labelled[['text', 'label']].copy()

In [12]:
# Convert Label to category
df_raw['label'] = pd.Categorical(df_raw.label)

In [13]:
# check!
df_raw.head(2)

,text,label
0,Happy th birthday to the @USNavy! The strength...,1
1,The greatest generation's investment in infras...,1


In [14]:
# safety copy
df = df_raw

### Train/Val/Test sets

In [15]:
# set lists
texts = list(df['text'])
labels = list(df['label'])

In [16]:
# Split the data in training and remaining dataset
x_train, x_rem, y_train, y_rem = train_test_split(texts, labels, train_size=0.8, random_state=seed)

# Split remaining data in testing and validation sets
x_val, x_test, y_val, y_test = train_test_split(x_rem,y_rem, test_size=0.5, random_state=seed)

print(len(x_train), len(x_val), len(x_test))

152245 19031 19031


## Load tokenizer and model

In [17]:
# Load tokenizer to process the text field:
tokenizer = XLMRobertaTokenizerFast.from_pretrained("xlm-roberta-base", use_fast=True)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [18]:
# call model
model = XLMRobertaForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=2) ## check n. of labels
model.to(device)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=2, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              

## Freeze!

In [19]:
# check names
for name, param in model.named_parameters():
    print(name)

classifier.dense.weight
classifier.dense.bias
classifier.out_proj.weight
classifier.out_proj.bias
roberta.embeddings.word_embeddings.weight
roberta.embeddings.token_type_embeddings.weight
roberta.embeddings.LayerNorm.weight
roberta.embeddings.LayerNorm.bias
roberta.embeddings.position_embeddings.weight
roberta.encoder.layer.0.attention.self.query.weight
roberta.encoder.layer.0.attention.self.query.bias
roberta.encoder.layer.0.attention.self.key.weight
roberta.encoder.layer.0.attention.self.key.bias
roberta.encoder.layer.0.attention.self.value.weight
roberta.encoder.layer.0.attention.self.value.bias
roberta.encoder.layer.0.attention.output.dense.weight
roberta.encoder.layer.0.attention.output.dense.bias
roberta.encoder.layer.0.attention.output.LayerNorm.weight
roberta.encoder.layer.0.attention.output.LayerNorm.bias
roberta.encoder.layer.0.intermediate.dense.weight
roberta.encoder.layer.0.intermediate.dense.bias
roberta.encoder.layer.0.output.dense.weight
roberta.encoder.layer.0.output.d

In [20]:
# Freeze layers except the classifier
for name, param in model.named_parameters():
    if 'classifier' not in name:  # Freeze all layers except the classifier
        param.requires_grad = False
    else:
        param.requires_grad = True  # Set requires_grad=True for the classifier parameters

In [21]:
# Check if layers are frozen
for name, param in model.named_parameters():
    print(f'{name}: requires_grad={param.requires_grad}')

classifier.dense.weight: requires_grad=True
classifier.dense.bias: requires_grad=True
classifier.out_proj.weight: requires_grad=True
classifier.out_proj.bias: requires_grad=True
roberta.embeddings.word_embeddings.weight: requires_grad=False
roberta.embeddings.token_type_embeddings.weight: requires_grad=False
roberta.embeddings.LayerNorm.weight: requires_grad=False
roberta.embeddings.LayerNorm.bias: requires_grad=False
roberta.embeddings.position_embeddings.weight: requires_grad=False
roberta.encoder.layer.0.attention.self.query.weight: requires_grad=False
roberta.encoder.layer.0.attention.self.query.bias: requires_grad=False
roberta.encoder.layer.0.attention.self.key.weight: requires_grad=False
roberta.encoder.layer.0.attention.self.key.bias: requires_grad=False
roberta.encoder.layer.0.attention.self.value.weight: requires_grad=False
roberta.encoder.layer.0.attention.self.value.bias: requires_grad=False
roberta.encoder.layer.0.attention.output.dense.weight: requires_grad=False
roberta.

In [ ]:
# @title
# Apply tokenisation
train_encodings = tokenizer(x_train, truncation=True, padding=True)
val_encodings = tokenizer(x_val, truncation=True, padding=True)
test_encodings = tokenizer(x_test, truncation=True, padding=True)

In [ ]:
# Set full datasets tokenised text & labels
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, y_train)
val_dataset = Dataset(val_encodings, y_val)
test_dataset = Dataset(test_encodings, y_test)

## Apply Sequence Classification (with Frozen Model)

In [ ]:
# define compute loss
def compute_loss(model, inputs, labels):
    outputs = model(**inputs)
    logits = outputs.logits.float()  # Convert logits to float data type
    loss_fn = nn.CrossEntropyLoss() #weight=class_weights
    loss = loss_fn(logits, labels)
    return loss

In [ ]:
# define metrics
def compute_metrics(p):
    pred, labels = p
    pred = np.argmax(pred, axis=1)
    accuracy = accuracy_score(y_true=labels, y_pred=pred)
    f1 = f1_score(y_true=labels, y_pred=pred, average='weighted')
    return {"accuracy": accuracy, "f1": f1}

In [ ]:
# Defining the TrainingArguments() arguments
args = TrainingArguments(
    output_dir=f"./results_political_tweets_xlm_roberta_{seed}_frozen",
    seed=seed,
    do_train=True,
    do_eval=True,
    evaluation_strategy ='steps',
    eval_steps = 50, # Evaluation and Save happens every 50 steps
    save_total_limit = 5, # Only last 5 models are saved. Older ones are deleted.
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=30,
    weight_decay=0.01,
    push_to_hub=False,
    metric_for_best_model = 'loss',
    load_best_model_at_end=True)

In [ ]:
# set Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=5)]
    )

In [ ]:
# run trainer (takes a while)
trainer.train()

### Evaluation

In [ ]:
# evaluate
trainer.evaluate()

In [ ]:
# evaluate on test
trainer.evaluate(test_dataset)

In [ ]:
# predict test set
predict = trainer.predict(test_dataset)

In [ ]:
# define parameters for validation
preds = predict.predictions
labels = predict.label_ids

In [ ]:
# create f-score metric
def f1_score_func(preds, labels):
    preds_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return f1_score(labels_flat, preds_flat, average='weighted')

In [ ]:
# flatten scores
preds_flat = np.argmax(preds, axis=1).flatten()
labels_flat = labels.flatten()

In [ ]:
# plot confusion matrix
print(confusion_matrix(labels_flat, preds_flat))
print(classification_report(labels_flat, preds_flat))

In [ ]:
# AUC test (Goal: >0.7)
fpr, tpr, thresholds = metrics.roc_curve(labels_flat, preds_flat, pos_label=1)
metrics.auc(fpr, tpr)

In [ ]:
# record seed for reproducibility
seed

## Save

In [ ]:
trainer.save_model(output_dir=f"./results_political_tweets_xlm_roberta_{seed}_frozen")

In [ ]:
trainer.save_state()